# Suite IDN — Identity: ownership and paths

A reference object's identity is its path: the route from a singleton to it through the relations that own it
(docs/OBJECT_IDENTITY.md). An adjacency marked `owner()` points back to the object's owner, through a relation whose
clauses make the route unambiguous; a key property may carry a naming pattern; a path is a nested value of
`Paths.OfRoot`, read off an object's ownership entries on demand. Step 4a of the store overhaul: the vocabulary, before
objects are identified by their paths.

In [ ]:
import { Builders as B, Paths, Plain, Schemas as S } from "@mbse/schemas/Framework";
import { LookupError } from "@mbse/schemas/Framework/Errors";
import { assert, equal, map, raises, same } from "./support.js";

const store: any = new B.Store();
const Text = store.Schemas.OfStr.as_schema(); // the store's own
const Directory = store.Schemas.OfObject((o: any) => o.name("wts.Directory").ref(true).singleton("wts.Features"));
const Definition = store.Schemas.OfObject((o: any) => o.name("wts.Definition").ref(true).properties([
  (t: any) => t.name("title").schema([Text]),
]));
const Clause = store.Schemas.OfObject((o: any) => o.name("wts.Clause").ref(true));
const Defines = store.Schemas.OfRelation((r: any) => (
  r
    .name("wts.Defines")
    .links([
      (t: any) => t.name("directory"),
      (t: any) => t.name("defn"),
    ])
    .properties([(t: any) => t.name("name").schema([Text]).pattern(String.raw`DEF-(\d+\d\d)`)])
    .uniques([["directory", "name"], ["defn"]])
));
const HasClause = store.Schemas.OfRelation((r: any) => (
  r
    .name("wts.HasClause")
    .links([
      (t: any) => t.name("defn"),
      (t: any) => t.name("clause"),
    ])
    .properties([(t: any) => t.name("name").schema([Text])])
    .uniques([["defn", "name"], ["clause"]])
));
store.Schemas.OfObject(Directory, (o: any) => o.adjacencies([(t: any) => t.name("dir").relation([Defines]).me("directory")]));
store.Schemas.OfObject(Definition, (o: any) => o.adjacencies([
  (t: any) => t.name("def").relation([Defines]).me("defn").owner(true),
  (t: any) => t.name("clauses").relation([HasClause]).me("defn"),
]));
store.Schemas.OfObject(Clause, (o: any) => o.adjacencies([(t: any) => t.relation([HasClause]).me("clause").owner(true)]));

## IDN-01 · An adjacency marked `owner()`: its relation has two links, and clauses by which the owned object determines its owner and the owner and keys determine the owned object

In [ ]:
{
  assert(Definition.adjacencies.get("def").owner === true && Directory.adjacencies.get("dir").owner === null);
  assert([Directory, Definition, Clause, Defines, HasClause].every((s) => s.validate().length === 0));
  /** An object schema owned through a relation of `links` with `uniques`: its validation. */
  const ownerOf = (uniques: string[][], links = ["up", "down"]): string[] => {
    const relation = store.Schemas.OfRelation((r: any) => r.links(links.map((n) => (t: any) => t.name(n))).uniques(uniques));
    return store.Schemas.OfObject((o: any) => o.ref(true).adjacencies([
      (t: any) => t.name("up").relation([relation]).me("down").owner(true),
    ])).validate();
  };
  assert(ownerOf([["up"], ["down"]]).length === 0); // keyless: the owner alone determines the owned object
  assert(same(ownerOf([["down"]]), ["adjacency 'up' is its object's owner: no unique clause has 'up' without 'down', so that "
    + "the owned object determines its owner"]));
  assert(same(ownerOf([["up"]]), ["adjacency 'up' is its object's owner: no unique clause has 'down' without 'up', so that "
    + "the owner and the keys determine the owned object"]));
  assert(same(ownerOf([["up"], ["down"]], ["up", "down", "side"]), [
    "adjacency 'up' is its object's owner: its relation has two links, the owner's and the owned's"]));
  raises(TypeError, () => store.Schemas.OfObject((o: any) => o.adjacencies([(t: any) => t.relation([Defines]).me("defn").owner("yes")])),
    "property 'owner' holds bool, got str");
}

## IDN-02 · A key property's naming pattern: the values it gives and reads, and what validation refuses

In [ ]:
{
  const name = Defines.entry.properties.get("name");
  assert(name.pattern === String.raw`DEF-(\d+\d\d)` && name.name_for(7) === "DEF-007" && name.name_for(1234) === "DEF-1234");
  assert(same([name.number_of("DEF-693"), name.number_of("DEF-69"), name.number_of("FEA-693")], [693, null, null]));
  assert(new S.OfProperty.Data({ name: "k", schema: Text }).number_of("DEF-693") === null); // no pattern
  const bad = store.Schemas.OfRelation((r: any) => (
    r
      .links([(t: any) => t.name("a"), (t: any) => t.name("b")])
      .properties([
        (t: any) => t.name("k").schema([Text]).pattern("DEF-.*"),
        (t: any) => t.name("n").schema([store.Schemas.OfInt]).pattern(String.raw`(\d+)`),
      ])
  ));
  assert(same(bad.validate(), [
    "property 'k': naming pattern 'DEF-.*' is not literal text around one group of digits, e.g. 'FEA-(\\d+\\d\\d)'",
    "property 'n': a naming pattern gives text: the property's schema is Schemas.OfStr"]));
}

## IDN-03 · Every store holds the path schemas and `Schemas.HasInstance`; the members of its own relations are owned

In [ ]:
{
  for (const name of ["Paths.OfRoot", "Paths.OfEntry", "Paths.OfTarget", "Paths.Next"]) {
    assert(store.registered(name).validate().length === 0 && store.framework(store.registered(name)));
  }
  assert(same([...store.Paths.OfRoot.as_schema().properties.keys()], ["singleton", "next"]));
  assert(same([...store.Paths.OfEntry.as_schema().properties.keys()], ["i", "o", "properties", "next"]));
  assert(same(store.Paths.Next.as_schema().branches.map((b: any) => b.name), ["entry", "target"]));
  const other: any = new B.Store();
  assert(other.Paths.OfRoot.as_schema() !== store.Paths.OfRoot.as_schema()); // each store's own
  const HasInstance = store.Schemas.HasInstance.as_schema();
  assert(same([...HasInstance.entry.links.keys()], ["schema", "instance"]) && same([...HasInstance.entry.properties.keys()], ["name"]));
  assert(store.Schemas.OfObject.as_schema().adjacencies.get("instances").relation === HasInstance);
  const owned = store.Schemas.OfProperty.as_schema().anonymous.filter((a: any) => a.owner).map((a: any) => `${a.relation.name} ${a.me}`);
  assert(same(owned, ["Schemas.HasProperty property"]));
  assert(store.Schemas.OfObject.as_schema().anonymous.filter((a: any) => a.relation.name === "Schemas.Defines").every((a: any) => a.owner));
}

## IDN-04 · An object's path on demand: read off its ownership entries back to a singleton, a nested value; refused where it has no owner, or more than one

In [ ]:
{
  const features = store.singleton("wts.Features");
  const definition = store.wts.Definition((d: any) => d.title("Has 88 keys"));
  store.wts.Directory(features, (f: any) => f.dir((a: any) => a.insert((e: any) => e.defn(definition).name("DEF-693"))));
  const clause = store.wts.Clause((c: any) => c);
  store.wts.Definition(definition, (d: any) => d.clauses((a: any) => a.insert((e: any) => e.clause(clause).name("C-1"))));
  const toPlain = (value: unknown) => Plain.ToPlain(store)(store.Paths.OfRoot.as_schema(), value);
  assert(equal(toPlain(Paths.of(definition)), map({ singleton: "wts.Features", next: { entry: {
    i: "dir", o: "def", properties: { name: "DEF-693" }, next: { target: {} } } } })));
  assert(equal(toPlain(Paths.of(clause)), map({ singleton: "wts.Features", next: { entry: {
    i: "dir", o: "def", properties: { name: "DEF-693" }, next: { entry: {
      i: "clauses", o: "", properties: { name: "C-1" }, next: { target: {} } } } } } }))); // an anonymous owner
  assert(equal(toPlain(Paths.of(features)), map({ singleton: "wts.Features", next: { target: {} } })));
  const loose = store.wts.Clause((c: any) => c);
  raises(LookupError, () => Paths.of(loose), "a 'wts.Clause' has no owner: its path is not known");
  const Twice = store.Schemas.OfObject((o: any) => o.name("wts.Twice").ref(true));
  const Holds = store.Schemas.OfRelation((r: any) => r.name("wts.Holds").links([(t: any) => t.name("a"), (t: any) => t.name("b")])
    .uniques([["a"], ["b"]]));
  store.Schemas.OfObject(Twice, (o: any) => o.adjacencies([
    (t: any) => t.name("up").relation([Holds]).me("b").owner(true),
    (t: any) => t.name("down").relation([Holds]).me("a"),
  ]));
  const [first, second] = [store.wts.Twice((t: any) => t), store.wts.Twice((t: any) => t)];
  store.wts.Twice(first, (t: any) => t.up((a: any) => a.insert((e: any) => e.a(second))));
  store.wts.Twice(second, (t: any) => t.up((a: any) => a.insert((e: any) => e.a(first))));
  raises(LookupError, () => Paths.of(first), "an ownership cycle: no singleton owns the object");
  const pair = [0, 1].map(() => store.Schemas.OfRelation((r: any) => r.links([(t: any) => t.name("a"), (t: any) => t.name("b")])
    .uniques([["a"], ["b"]])));
  store.Schemas.OfObject(Directory, (o: any) => o.adjacencies([(t: any) => t.relation([pair[0]]).me("a"),
    (t: any) => t.name("rights").relation([pair[1]]).me("a")]));
  store.Schemas.OfObject((o: any) => o.name("wts.Split").ref(true).adjacencies([
    (t: any) => t.name("left").relation([pair[0]]).me("b").owner(true),
    (t: any) => t.name("right").relation([pair[1]]).me("b").owner(true),
  ]));
  const split = store.wts.Split((s: any) => s.left((a: any) => a.insert((e: any) => e.a(features))));
  assert(equal(toPlain(Paths.of(split)), map({ singleton: "wts.Features", next: { entry: {
    i: "", o: "left", properties: {}, next: { target: {} } } } }))); // the owner's adjacency anonymous
  store.wts.Split(split, (s: any) => s.right((a: any) => a.insert((e: any) => e.a(features))));
  raises(LookupError, () => Paths.of(split), "a 'wts.Split' has more than one owner: its path is not known");
}